# Medical DAPT - DoRA ablation study, **Qwen3.5-4B-Base**

**Three files, one job each.** This notebook is the narrative and the driver;
it holds no numbers and no logic.

| file | holds | change it when |
|---|---|---|
| `configs/base.json` | every number: model, paths, budget, baseline, the grid, the data mix, the ruler tolerance | you want a different experiment |
| `dapt_lib.py` | every function: budget math, model build, data, trainer, probe, driver, ruler | you want different behaviour |
| this notebook | the why, and one-line calls into the above | (rarely -- only to reorder steps) |

A **variant** never forks the notebook: `open_run(CFG_PATH, overrides="ctx4096.json")`
deep-merges `configs/ctx4096.json` (which holds only what it changes) over the
base. Dicts merge key-by-key; arrays are replaced wholesale, so a variant that
changes the grid restates the whole array.

Every arm stamps the **resolved config and its sha256** into its `status.json`
and into `results.csv`, so a results row is self-describing months later.

`dapt_lib.py` imports nothing heavy at module level, so `budget`, `selftest` and
`ruler` run on a bare interpreter -- which is how they are testable off-GPU:

```bash
python3 dapt_lib.py budget      # the cost table, no torch needed
python3 dapt_lib.py selftest    # 12 checks: config, budget math, status/CSV contract
python3 dapt_lib.py ruler       # scores whatever arms have finished
```

---

**Model swap.** `Qwen/Qwen3.8-27B` -> `Qwen/Qwen3.5-4B-Base` (4.66B total =
4.35B text tower + 0.31B ViT, Apache-2.0, ungated).  This is close to a
drop-in, because it is the same architecture family:

- `qwen3_5` / `Qwen3_5ForConditionalGeneration`, hybrid **3 linear
  (Gated-DeltaNet) : 1 full attention**, `full_attention_interval=4`,
  `head_dim=256`, `hidden_size=2560`, 32 layers, vocab **248,320** - so
  `target_modules`, `finetune_vision_layers=False`, the bundled ViT and the
  text-tokenizer path are all unchanged;
- the same adapter surface: `use_rslora=False` (conflicts with DoRA),
  `modules_to_save=["embed_tokens","lm_head"]`, `load_in_4bit=False`.

**Three things that genuinely differ, all handled in `dapt_lib.py`.**

1. `transformers` **must be v5** - Unsloth's floor for `qwen3_5` is `5.2.0`
   and its ceiling is `5.17.0` (latest on PyPI is 5.19.0, i.e. *too new*).  The
   old `transformers>=4.40.0` pin sat below the floor.  The install cell pins
   the window; `verify_env()` gates on it and refuses to continue below 5.2.
   Installs must also keep `datasets<5.0.0` (latest is 5.1.0, which unsloth
   rejects) and `trl<=1.13.0` (latest is 1.15.0).
2. **`tie_word_embeddings=True` on the 4B; the 27B sets it `False`.**  PEFT's
   `modules_to_save` deep-copies every wrapped module, so both stay trainable
   but **the embedding/head tie is broken** - the head no longer follows the
   embedding for free.  A checkpoint is therefore ~2.5 GB (2 x 636M params
   bf16), not the 3.5 GB the 27B paid.  `build_model()` prints exactly what
   happened, and **asserts** it: wrapping only one of the pair would leave the
   other frozen against a stale copy, which is worse.  PEFT ships the opt-in
   fix (`ensure_weight_tying=True`: one trainable embedding/head tensor, the
   head wired to it, ~1.2 GB checkpoints) and it is exposed as the baseline key
   `ensure_weight_tying` - **false here on purpose**, because it changes what
   is trainable and therefore belongs in the grid as its own arm, not in the
   baseline as a silent change.
3. **bf16 is mandatory and QLoRA is discouraged.**  Unsloth lists `qwen3_5` in
   `FORCE_FLOAT32` - "GDN layers produce NaN grad norms in float16 training" -
   and its docs advise against 4-bit for Qwen3.5.  The config keeps
   `bf16=True`, `fp16=False`; `build_model()` never loads 4-bit.

**Three free wins** on the 4B: Unsloth quotes ~10 GB VRAM for bf16 LoRA
(96 GB card here), model reload drops to ~1.5 min (`overheads.model_load_min`),
and `causal_conv1d` + the bundled flash-linear-attention kernels stay as they
are (the GDN conv needs the former; Unsloth vendors the latter, so the explicit
`fla` install is belt-and-braces).

---

**Design.** One-factor-at-a-time around a fixed baseline, so every arm reads
directly against arm `B`. **6 arms.** The baseline itself moved: the original
`r=128` anchor was **4.93% of the base model** on its own, so it is gone rather
than ablated around.

| id | varies | value | adapter % of base |
|---|---|---|---|
| **B** | - (baseline) | DoRA r=32, a=64, LR 1e-4, embLR 1e-5, drop 0, len 2048 | 1.25% |
| R16 | rank | r=16 (a=32) | 0.63% |
| LO | method | LoRA instead of DoRA (r=32) | 1.23% |
| LR3e4 | learning rate | 3e-4 (embLR 3e-5) | 1.25% |
| EMB1 | embedding LR | no decoupling (embLR = LR) | 1.25% |
| ADONLY | trainable surface | `modules_to_save=None` - embed/head **frozen** | 1.25% |

**`ADONLY` is the one arm that is not a one-factor tweak.** It changes *what is
trainable*, so read it as its own recipe rather than as a delta from `B`. It
exists because of the numbers in the next section: `modules_to_save` makes
1.271B of the 1.329B trainable parameters the embed/head pair. Freezing that
pair takes trainable parameters from **1.329B to 58.1M (22.9x)**, grads plus
8-bit optimizer state from **~5.3 GB to ~0.23 GB**, checkpoints from **~2.5 GB
to ~0.12 GB**, and lifts data exposure from **0.034 to 0.785 tokens per
trainable parameter**. The tie stays *intact* in this arm (nothing is wrapped),
which `build_model()` asserts. If `ADONLY` matches `B` on held-out domain loss,
the cheaper recipe is the better one and the confirm stage changes shape - that
is the whole reason it is worth one slot.

**What was cut, and why.**

- **`r=128` and `r=256` - removed entirely, not ablated.** They are 4.93% and
  9.83% of the base model for a single adapter, i.e. 2.5x and 5x the "keep it
  under ~1-2%" line, and every study that tested rank saturation puts the
  payoff at 8-32.  The DoRA paper reports its own LLaMA-7B setup at **2.31%
  (LoRA) / 2.33% (DoRA)** and argues the method exists partly to *reduce* the
  rank you need: its half-rank variant beats full-rank LoRA by 2.8 points
  "despite having only half as many trainable parameters ... reducing the need
  for a higher rank to surpass LoRA".  Paying 4x the parameters to find out
  whether rank 128 beats rank 32 was never the question worth 25 h.
- `R64` - 2.47%, already over the line for a question whose answer is known.
- `R16` **kept** (one arm, 0.63%): it is the free direction, and it is the
  half of DoRA-vs-double-rank-LoRA the paper's `DoRA-dagger` result predicts.
- `LR3e5` - redundant direction.  1e-4 -> 3e-4 brackets the DoRA/LoRA default
  of ~2e-4 with one arm instead of two.
- `DR05` - lora_dropout is not a lever at this token budget.
- `LEN4096` - context length is a deployment choice, not an adaptation
  question.  It lives in `configs/ctx4096.json` as a variant that adds one 4k
  arm; for the same token budget that arm gets half the steps anyway.

**Adapter parameter budget on this model** (4.66B params; the 7 targeted
projections across all 32 layers; DoRA's magnitude vectors add 0.95M = 0.02%
on top of whatever `r` gives you):

| r | adapter params | % of model | in this grid? |
|---|---|---|---|
| 16 | 29.5M | 0.63% | R16 |
| 32 | 58.1M | 1.25% | **B**, LO, LR3e4, EMB1 |
| 64 | 115.2M | 2.47% | cut |
| 128 | 229.5M | 4.93% | cut (old baseline) |
| 256 | 458.1M | 9.83% | cut |

Every adapter arm sits at or under **1.25%** of the base, against the DoRA
paper's own 2.31%/2.33% LLaMA-7B figure.  One honest counterweight: the adapter
is still not where your trainable parameters live.  `modules_to_save` on
`embed_tokens` + `lm_head` is **1.271B = 27.3% of the model** by itself, so at
r=32 total trainable is **1.329B = 28.5%** and the adapter is only **4.4%** of
it.  At 45.6M tokens/arm that is **0.034 tokens per trainable param** -
data-starved, which is the regime where low rank wins.  `EMB1` and `ADONLY`
both attack this budget from different ends: `EMB1` changes how fast the 1.271B
moves, `ADONLY` deletes it entirely (0.785 tokens/param).

Rank arms hold effective scaling constant (`alpha = 2*r`), because rsLoRA is
**not** usable with DoRA: Unsloth states LoRA+/DoRA "does not interact well with
rsLoRA".  `check_config()` asserts the baseline keeps `alpha == 2r`.

**Budget is defined in TOKENS, not epochs.** A streaming interleaved mix has no
well-defined epoch; fixing a token budget is what makes the arms comparable.
`budget.seq_len` is the reference length the budget is defined at -- to change
context, set an *arm's* `max_length`, never `seq_len` (which would silently
rescale tokens/arm; `check_config()` warns if they diverge).

**Steps per arm - what the config is set to, and why.** At ~12 s/step
(2,731 tok/s, 9.83M tok/h) and 0.36 h of fixed per-arm overhead (reload + 8
eval passes):

| wall h/arm | steps/arm | tokens/arm | tok/param (4.35B) | units/arm |
|---|---|---|---|---|
| 1.0 | 192 | 6.3M | 0.0015 | 9 |
| 3.0 | 792 | 26.0M | 0.0060 | 26 |
| **5.0 (set here)** | **1392** | **45.6M** | **0.0105** | **44** |
| 10.0 | 2892 | 94.8M | 0.0218 | 87 |

6 arms x 5.0 h = **30 h ~ 261 Colab units ~ 3.2 sessions** (9.5 h each).  Cutting
10 arms to 6 is what pays for the extra 2 h/arm.  `budget.total_budget_hours` is
45.0 so the two seed replicates of `B` below (10 h) still fit under the ceiling
without editing anything else.

Honest scale check: **0.0105 tokens/param is a domain nudge, not DAPT**
(canonical DAPT is 1-7 tok/param).  This grid *ranks arms*; it does not by
itself prove medical adaptation.  `phase0()` is the separate, cheaper experiment
that answers that.

**Do this first, or the grid is unreadable.** Seed replicates of the baseline
are the noise floor - not ablation arms. Append to the `grid` array in
`configs/base.json`:

```json
{"id": "B_s1", "desc": "baseline, seed 1", "overrides": {"seed": 1}},
{"id": "B_s2", "desc": "baseline, seed 2", "overrides": {"seed": 2}},
```

10 h / 87 units. If the three `B` runs scatter by more than the effects you are
chasing (rank, LR, embedding LR), the grid cannot rank anything and the fix is
more tokens, not more arms.

**Keep/throw ruler.** Every arm logs its own step-0 held-out losses, so the
forgetting veto is measured against that arm's own starting point.
Verdict = best held-out **domain** loss, vetoed by a **retention regression** on
the general/reasoning guards (FineWeb-Edu, Orca-Math); the allowed rise is
`ruler.retention_tol` = 0.05.

## 1. Install

Qwen3.5 needs the transformers v5 window; unsloth owns it. Order matters:
  unsloth first so its metadata pins transformers/trl/peft/datasets, then the
  explicit line can only tighten.

In [ ]:
# Order matters. unsloth is installed FIRST so its own metadata pins
# transformers/trl/peft/datasets, then the explicit line can only tighten.
# Pins that are load-bearing for qwen3_5:
#   transformers >= 5.2.0   (below this the qwen3_5 arch does not exist)
#   transformers <= 5.17.0  (5.19.0 is on PyPI and is outside unsloth's range)
#   datasets     <  5.0.0   (5.1.0 on PyPI is rejected by unsloth)
#   trl          <= 1.13.0  (1.15.0 on PyPI is outside unsloth's range)
#   peft         >= 0.18.0
%pip install -q -U unsloth unsloth_zoo
%pip install -q -U "transformers>=5.2.0,<=5.17.0" "peft>=0.18.0" "trl>=0.18.2,<=1.13.0" "datasets>=3.4.1,<5.0.0" accelerate bitsandbytes ninja packaging torchao
%pip install -q https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.4/flash_attn-2.8.3%2Bcu130torch2.11-cp313-cp313-linux_x86_64.whl
# Unsloth vendors the flash-linear-attention kernels for gated-deltanet layers;
# this install is belt-and-braces (and what the 27B run used). Cell 5 reports
# whether the fast kernels actually came up.
%pip install -q -U flash-linear-attention
%pip install -q cut-cross-entropy
%env HF_XET_HIGH_PERFORMANCE=1

## 2. HF auth, Drive, and a fresh compile cache

Drive holds the results tree, so it is mounted before anything writes to it.
The **code** is not here any more -- cell 4 clones that from GitHub -- so this
cell is only credentials plus the mount.

In [ ]:
import os
import shutil
from dotenv import load_dotenv
import huggingface_hub
from google.colab import drive

load_dotenv()
huggingface_hub.login(token=os.getenv("HF_TOKEN"))

drive.mount("/content/drive")

# Fresh compile cache every session: unsloth caches compiled kernels, and a
# stale one is a classic "it worked yesterday" failure.
shutil.rmtree("/content/unsloth_compiled_cache", ignore_errors=True)
print("drive mounted -- the ablation root comes from the config in cell 4")

## 3. causal_conv1d wheel

The GDN layers need the prebuilt `causal_conv1d`; the G4 is cu130/torch2.11.
`--no-deps`: the wheel must not drag its own torch in.

In [ ]:
%pip install -q --no-deps "/content/drive/MyDrive/wheels/causal_conv1d-1.7.0-cp313-cp313-linux_x86_64.whl"

## 4. Bootstrap: clone the code from GitHub -> open the run

`run` is the only piece of session state. It carries the config, the resolved
budget, the arm list and the data streams, so a kernel restart costs you one
re-run of this cell rather than a re-typed control panel.
  
The **code** comes from GitHub (`med-dapt-qwen35-ablation`), cloned once per
session: `dapt_lib.py` + `configs/`. The **results** still go to Drive, because
that is what has to survive the session -- `run.root` comes from
`ablation_root` in the config, and the driver resumes off the `status.json`
files it finds there. Re-running this cell pulls the latest code instead of
re-cloning.

In [ ]:
# --- the code lives in a GitHub repo: a fresh session clones it -------------
REPO      = "https://github.com/adi274903/med-dapt-qwen35-ablation.git"
DAPT_HOME = "/content/med-dapt-qwen35-ablation"          # dapt_lib.py + configs/
CFG_PATH  = DAPT_HOME + "/configs/base.json"
OVERRIDES = None            # e.g. "ctx4096.json" -- resolved next to base.json

import importlib
import os
import subprocess
import sys

# rerun-safe: pull if already cloned, clone once otherwise. --depth 1 keeps
# it to a few hundred KB; the results themselves still go to Drive.
if os.path.isdir(os.path.join(DAPT_HOME, ".git")):
    subprocess.run(["git", "-C", DAPT_HOME, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, DAPT_HOME], check=True)

if DAPT_HOME not in sys.path:
    sys.path.insert(0, DAPT_HOME)
import dapt_lib
importlib.reload(dapt_lib)      # an edit to dapt_lib.py lands without a restart

run = dapt_lib.open_run(CFG_PATH, overrides=OVERRIDES)
os.makedirs(run.root, exist_ok=True)        # the ablation tree, from the config

print("model     :", run.cfg["model_id"])
print("config    :", run.config_path, " (hash %s)" % run.cfg_hash)
print("overrides :", run.cfg.get("_overrides_path", "none"))
print("root      :", run.root)
print("arms      :", ", ".join(a["arm_id"] for a in run.arms))
print("")
print("nothing below this cell needs editing: change configs/base.json instead.")

## 5. Environment gates

Once per session. The three things that silently ruin a run: transformers
outside the v5 window, fp16 instead of bf16, and a QLoRA path that Unsloth
advises against for this model.

In [ ]:
dapt_lib.verify_env(run)

## 6. Budget

Nothing to edit here -- this is a readout of `configs/base.json`. It prints the
steps, the tokens, the honest per-arm overhead and the whole grid's cost, and it
shouts if `sec_per_step` is still the placeholder.

In [ ]:
arms = dapt_lib.print_budget(run)

## 7. Data

One streaming interleaved mix plus a per-source holdout, built once and reused
by every arm so all arms see the same data in the same order. Source list, mix
and holdout size are all in the config.

In [ ]:
dapt_lib.build_data(run)

## 8. Probe -- measure s/step on THIS box before committing the grid

~5 min. Skipping it does not error: a stale `sec_per_step` silently rescales
the token budget, and the token budget is the only thing that makes the arms
comparable. Paste the reported number into `budget.sec_per_step` in
`configs/base.json`, then re-run **cell 4** and **cell 6** before the grid.

In [ ]:
sec_per_step = dapt_lib.probe(run)

## 9. Run the grid

Pending arms only, fresh base + fresh adapters per arm, stopping on either the
grid budget or the session budget. Interrupting is safe: re-run it in a fresh
session and it continues where it stopped.

In [ ]:
dapt_lib.run_grid(run)

## 10. TensorBoard

`run.root` is the Qwen3.5-4B tree, so this never mixes with the old 27B logs.

In [ ]:
%load_ext tensorboard
# run.root is interpolated from the config = the 4B tree, never the old
# /content/drive/MyDrive/Med_Qwen3.8_27B/ablation
%tensorboard --logdir {run.root}

## 11. Keep/throw ruler

Scores every finished arm: best held-out domain loss, vetoed by a retention
regression against that arm's own step-0 baseline. Writes `ranking.csv` next to
`results.csv`. Returns the winning arm id as `best`, which cell 12 uses.

In [ ]:
best = dapt_lib.ruler(run)

## 12. Validate the winner's adapters

Deliberately NOT part of the grid -- this is validation, not ablation, and it
needs the arm to have been checkpointed (`saving.save_models: true`, or the arm
id listed in `saving.save_arms`).
  
`modules_to_save` writes the FULL `embed_tokens` + `lm_head`. On this tied base
PEFT's wrapper unties the copies, so expect ~2.5 GB per checkpoint. If `best` is
`ADONLY` there is no `modules_to_save` in the state dict at all: the counts
print 0 and the tie check prints False.

In [ ]:
_ = dapt_lib.verify_adapters(run, best)

## 13. Phase 0 (optional) -- the does-DAPT-help curve

The grid answers *which config ranks best*. It does **not** answer *did the
medical mix move this base*: a 5 h arm sees 45.6M tokens = 0.0105 tokens/param,
and canonical DAPT is 1-7 tok/param.
  
`phase0()` runs **only the baseline** at growing step counts and prints held-out
domain mean vs tokens. 300 -> 8000 steps spans 9.8M -> 262M tokens (0.002 ->
0.060 tok/param) and ~1.4 -> 27.0 h of wall clock; the bend in that curve is the
answer, and it also tells you which grid budget is worth spending. Runs >=
`saving`-forced steps checkpoint and can resume across sessions. It writes to its
own root, so it can never collide with the grid, and its rows carry
`phase="phase0"` in `results.csv`. Flip `phase0.run` to `true` in the config to
run it.
  
Cheaper alternative worth considering first: the same curve on
**Qwen3.5-0.8B-Base**. Same architecture family, same tokenizer, ~5.4x the
tokens/param for the same wall clock, and the recipe ranking transfers.

In [ ]:
_ = dapt_lib.phase0(run)